# 09. 규칙 기반 + ML 결합 — 튜터 권고를 실측으로 검증하다

중간보고 튜터 피드백: *“극단적 불균형에서 단일 모델만으로 전 클래스를 커버하는 데 구조적 한계가 있다 — 규칙 기반 알고리즘과 ML 결합 시도를 권장한다.”*

이 노트북은 그 권고를 **실제로 구현해 보고, 이득이 없음을 수치로 확인한** 기록이다.

> **출처** — `원본/최종_3차_통합분석.ipynb` (코드 셀 **1개**, 29,946자)
>
> 원본은 `RUN_PART1`~`RUN_PART4` 플래그로 5개 파트를 한 셀에서 켜고 끄는 구조였다.
> 이 저장소는 그 플래그 경계를 그대로 살려 파트별 노트북으로 분리하고,
> **원본 실행 출력(표·그래프)을 해당 셀에 그대로 옮겨 붙였다.**
> 따라서 데이터가 없어도 결과를 볼 수 있고, 데이터가 있으면 그대로 재실행된다.



### 재현 조건

| 항목 | 값 |
|---|---|
| 입력 | `data/X_tr.parquet` · `X_va.parquet` · `y_tr.parquet` · `y_va.parquet` (**3차 전처리**) |
| 형상 | train `(96,140 × 58)` / valid `(23,860 × 58)` |
| 정상 클래스 | `12` (= `m`) |
| 모델 | `LGBMClassifier` · `n_estimators=300` · `class_weight="balanced"` · `random_state=42` |
| 학습 시간 | 약 13초 (4스레드) |

> `data/` 는 용량(141MB) 때문에 저장소에 포함하지 않았다 — `data/README.md` 참고.
> 아래 셀의 출력은 **원본 실행 결과 그대로**다.

> ⚠️ 원본의 `if RUN_PARTn:` 가드는 제거하고 들여쓰기를 풀었다.
> 파트별 노트북이므로 그 파트는 항상 실행된다 — **로직은 원본과 동일**하다.


## 0. 공통 설정 · 데이터 로드 · 모델 학습

다섯 개 노트북이 모두 이 블록으로 시작한다. 원본 통합 셀의 `[0]` 구획이다.


In [ ]:
# [0] 설정 · 데이터 로드 · 모델 학습 (여기부터 순서대로 실행)
import warnings, time, logging
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ===== 설정 =====
DATA_DIR = "../../data"        # 3차 전처리 parquet 폴더 (원본 실행 경로:
                               #   /mnt/c/Users/dkstj/Desktop/test_uv/3차 전처리 파일)
SEED = 42
LGBM_THREADS = 4
N_ESTIMATORS = 300             # 느리면 150~200 (수치가 달라짐)

RUN_PART1 = True               # 산출물 4종
RUN_COMPARE = True             # 모델 비교(한방 기본/임계값 · 2계층 기본/임계값) — 2계층 학습 포함
RUN_PART2 = True               # 비용기반 임계값
RUN_PART3 = True               # 규칙기반 + ML 결합
RUN_PART4 = True               # 생성 AI 합성데이터 증강 비교 (느림: sdv 필요)

LEAK_COLS = ["is_fraud", "fraud_type", "label", "y"]      # 정답 라벨 자동 제거
LOOKAHEAD_SUSPECT = ["Account_one_month_max_amount", "Account_one_month_std_dev",
                     "Account_dawn_one_month_max_amount", "Account_dawn_one_month_std_dev",
                     "Amount_vs_monthly_max_ratio"]        # 남아 있으면 경고만

# 비용 가정 (팀 합의값으로 교체)
FN_COST = 17_000_000
FP_SCENARIOS = {"A (FP 5k)": 5_000, "B (FP 10k)": 10_000, "C (FP 50k)": 50_000}
MACRO_MIN = 0.60               # 최종 채택 조건: Macro-F1 최소기준선
AMOUNT_COL = "Transaction_Amount_abs"

# 임계값 격자
THR_FULL = [0.9, 0.7, 0.5, 0.3, 0.2, 0.1, 0.05, 0.03, 0.02, 0.01,
            0.005, 0.003, 0.002, 0.001, 0.0005, 0.0002, 0.0]
THR_BARS = [0.0, 0.0002, 0.0005, 0.001, 0.002, 0.003, 0.005, 0.008, 0.01, 0.02, 0.03]
THR_COLLAPSE = [0.0, 0.00001, 0.00002, 0.00005, 0.0001, 0.00015, 0.0002]

SHAP_SAMPLE = 2000
TOP_FEATURES = 15

# [3] 하이브리드
RULE_MIN_SUPPORT = 15
PRECISION_SCAN = [0.85, 0.70, 0.50, 0.30]
CONT_Q_HI, CONT_Q_LO = [0.90, 0.95, 0.99], [0.01, 0.05, 0.10]
VERIFIED_COMPOSITES = ["large_deposit_and_remote_control", "unused_terminal_and_internet",
                       "limit_check_then_transfer", "suspended_recipient_and_withdrawal",
                       "new_recipient_and_large_amount", "weak_signal_composite_score"]
BOOST_WEIGHT = 0.30

# [4] 증강
TRAIN_SUBSAMPLE_NORMAL = 20000
TARGET_PER_CLASS = 500
GEN_EPOCHS = 300

LGBM_PARAMS = dict(n_estimators=N_ESTIMATORS, learning_rate=0.05, num_leaves=31,
                   subsample=0.9, colsample_bytree=0.9, class_weight="balanced",
                   random_state=SEED, n_jobs=LGBM_THREADS, verbose=-1)
lab = lambda i: chr(ord("a") + int(i))
np.random.seed(SEED)

In [ ]:
# ===== 표시 유틸 =====
def _setup_plt():
    import matplotlib
    try:
        ip = get_ipython()  # type: ignore
        if ip is not None:
            ip.run_line_magic("matplotlib", "inline")
    except Exception:
        pass
    import matplotlib.pyplot as plt
    logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
    matplotlib.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    return plt


def _table(df, caption=""):
    """VS Code/Jupyter면 표로 렌더링(출력 잘림 방지), 아니면 텍스트."""
    if caption:
        print(f"\n[{caption}]")
    try:
        ip = get_ipython()  # type: ignore
        if ip is not None:
            from IPython.display import display
            display(df); return
    except Exception:
        pass
    print(df.to_string(index=False))


def _head(title):
    print("\n" + "=" * 74); print(f" {title}"); print("=" * 74)

In [ ]:
# ===== 데이터 로드 =====
from lightgbm import LGBMClassifier
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)

_d = Path(DATA_DIR)
X_tr = pd.read_parquet(_d / "X_tr.parquet").reset_index(drop=True)
X_va = pd.read_parquet(_d / "X_va.parquet").reset_index(drop=True)
y_tr = pd.read_parquet(_d / "y_tr.parquet").iloc[:, 0].astype(int).reset_index(drop=True)
y_va = pd.read_parquet(_d / "y_va.parquet").iloc[:, 0].astype(int).reset_index(drop=True)

_drop = [c for c in X_tr.columns if c.lower() in [s.lower() for s in LEAK_COLS]]
if _drop:
    print(f"[누수 제거] 정답 라벨 {_drop} 자동 제외")
    X_tr = X_tr.drop(columns=_drop); X_va = X_va.drop(columns=_drop)
_rest = [c for c in LOOKAHEAD_SUSPECT if c in X_tr.columns]
if _rest:
    print(f"[경고] look-ahead 의심 피처 잔존: {_rest}")
    print("       → as-of(시점까지 누적) 재설계 여부 전처리팀 확인 필요")

normal_idx = int(y_tr.value_counts().idxmax())
classes = sorted(y_tr.unique()); labels = [lab(c) for c in classes]
fcols = list(X_tr.columns)
amount = X_va[AMOUNT_COL].values.copy() if AMOUNT_COL in X_va.columns else None
basis = f"{X_tr.shape[1]} features | valid {len(y_va):,} rows (hold-out)"
basis_kr = f"피처 {X_tr.shape[1]}개 · valid 홀드아웃 {len(y_va):,}행 · 트리 {N_ESTIMATORS}"

_head(f"3차 전처리 통합 분석 — {basis_kr}")
print(f"[데이터] train {X_tr.shape} / valid {X_va.shape} | 정상={normal_idx}({lab(normal_idx)})")


# ===== 모델 학습 (1회, 전 파트 공유) =====
print("\n[학습] LightGBM 13-class ...")
_t = time.time()
model = LGBMClassifier(**LGBM_PARAMS).fit(X_tr, y_tr)
pred = model.predict(X_va)
macro = f1_score(y_va, pred, average="macro")
print(f"[검증] {round(time.time()-_t,1)}s | Macro-F1={macro:.4f} | "
      f"Weighted-F1={f1_score(y_va, pred, average='weighted'):.4f} | "
      f"Accuracy={accuracy_score(y_va, pred):.4f}")
print("  ※ Accuracy는 정상 99%라 무의미 → Macro-F1로 판단")

rep = classification_report(y_va, pred, labels=classes, target_names=labels,
                            output_dict=True, zero_division=0)
proba = model.predict_proba(X_va)
_cls = list(model.classes_); _npos = _cls.index(normal_idx)
risk = 1.0 - proba[:, _npos]                       # 위험점수 = 1 − P(정상)
_fp = proba.copy(); _fp[:, _npos] = -1.0
best_fraud = np.asarray(_cls)[_fp.argmax(axis=1)]  # 정상 제외 최고확률 유형
y_arr = np.asarray(y_va); yb = (y_arr != normal_idx)

[경고] look-ahead 의심 피처 잔존: ['Account_one_month_max_amount', 'Account_one_month_std_dev', 'Account_dawn_one_month_max_amount', 'Account_dawn_one_month_std_dev', 'Amount_vs_monthly_max_ratio']
       → as-of(시점까지 누적) 재설계 여부 전처리팀 확인 필요

 3차 전처리 통합 분석 — 피처 58개 · valid 홀드아웃 23,860행 · 트리 300
[데이터] train (96140, 58) / valid (23860, 58) | 정상=12(m)

[학습] LightGBM 13-class ...
[검증] 13.3s | Macro-F1=0.6138 | Weighted-F1=0.9942 | Accuracy=0.9949
  ※ Accuracy는 정상 99%라 무의미 → Macro-F1로 판단


위험점수 `risk = 1 − P(정상 m)` 기준 임계값 판정 헬퍼. 임계값을 쓰는 파트에서 공통으로 사용한다.


In [ ]:
def thr_predict(thr):
    """위험점수 ≥ thr → 사기(최고확률 유형), 아니면 정상."""
    p = np.full(len(y_arr), normal_idx); m = risk >= thr
    p[m] = best_fraud[m]; return p, m


def thr_scan(grid):
    rows = []
    for t in grid:
        p, m = thr_predict(t)
        fn = int((yb & ~m).sum()); fp = int((~yb & m).sum())
        r = dict(thr=t, macro=round(f1_score(y_arr, p, average="macro"), 4), FN=fn, FP=fp)
        for k, v in FP_SCENARIOS.items():
            r[k] = round((fn * FN_COST + fp * v) / 1e8, 3)
        if amount is not None:
            r["FN실제금액(억)"] = round(float(amount[yb & ~m].sum()) / 1e8, 2)
            r["FN고정값(억)"] = round(fn * FN_COST / 1e8, 2)
        rows.append(r)
    return pd.DataFrame(rows)

## 3-1. 규칙 후보 탐색 — atom 조합으로 고정밀 규칙을 찾는다

`atom` = 단일 조건(`검증된 파생변수==1` / `범주형==값` / `연속형 분위수 이상·이하`).
유형별로 상위 15개 atom 의 2~3개 조합을 전수 탐색해 precision 최고 규칙을 찾는다.


In [ ]:
_head("[3] 규칙기반 + ML 결합 (하이브리드)")
plt = _setup_plt()

def _atom_mask(X, a):
    f, op, v = a; col = X[f].values
    return col == v if op == "==" else (col >= v if op == ">=" else col <= v)

def _cond_mask(X, atoms):
    m = np.ones(len(X), dtype=bool)
    for a in atoms:
        m &= _atom_mask(X, a)
    return m

def _desc(atoms):
    return " & ".join(f"{a[0]}{a[1]}{a[2]:g}" for a in atoms)

# 후보 atom: 검증 파생변수 + 범주형 단일 + 연속형 분위수
atoms = [(c, "==", 1) for c in VERIFIED_COMPOSITES if c in X_tr.columns]
for c in fcols:
    sc_ = X_tr[c]
    if sc_.nunique() <= 10:
        for v in sorted(sc_.unique()):
            m = (sc_ == v).values
            if 30 <= m.sum() <= len(X_tr) * 0.5:
                atoms.append((c, "==", int(v) if float(v).is_integer() else float(v)))
    else:
        for q in CONT_Q_HI:
            atoms.append((c, ">=", float(sc_.quantile(q))))
        for q in CONT_Q_LO:
            atoms.append((c, "<=", float(sc_.quantile(q))))
print(f"[후보] atom {len(atoms)}개 (검증파생 + 범주형 + 연속형분위수) → 유형별 조합 탐색")

ytr_arr = np.asarray(y_tr)
cands, per_best = [], {}
for t in [c for c in classes if c != normal_idx]:
    yt = (ytr_arr == t); tot = int(yt.sum())
    scored = []
    for a in atoms:
        m = _atom_mask(X_tr, a); hit = int((yt & m).sum())
        if hit >= max(tot * 0.2, 5):
            scored.append((hit / m.sum(), a, m))
    scored.sort(key=lambda x: -x[0]); top = scored[:15]
    best_r = (0.0, None, 0)
    for i in range(len(top)):
        for j in range(i + 1, len(top)):
            m2 = top[i][2] & top[j][2]
            if m2.sum() >= RULE_MIN_SUPPORT:
                p2 = float((yt & m2).sum() / m2.sum())
                cands.append(([top[i][1], top[j][1]], t, p2, int(m2.sum())))
                if p2 > best_r[0]: best_r = (p2, [top[i][1], top[j][1]], int(m2.sum()))
                for l in range(j + 1, len(top)):
                    m3 = m2 & top[l][2]
                    if m3.sum() >= RULE_MIN_SUPPORT:
                        p3 = float((yt & m3).sum() / m3.sum())
                        cands.append(([top[i][1], top[j][1], top[l][1]], t, p3, int(m3.sum())))
                        if p3 > best_r[0]:
                            best_r = (p3, [top[i][1], top[j][1], top[l][1]], int(m3.sum()))
    per_best[lab(t)] = best_r

_table(pd.DataFrame([{"유형": k, "최고 precision": round(v[0], 3), "hits": v[2],
                      "조건": (_desc(v[1]) if v[1] else "")[:70]}
                     for k, v in per_best.items()]),
       "유형별 최고 달성 가능 규칙 (train 기준)")


 [3] 규칙기반 + ML 결합 (하이브리드)
[후보] atom 181개 (검증파생 + 범주형 + 연속형분위수) → 유형별 조합 탐색

[유형별 최고 달성 가능 규칙 (train 기준)]


,유형,최고 precision,hits,조건
0,a,0.933,15,Distance>=301.348 & Time_difference_seconds<=1...
1,b,0.211,19,Amount_vs_remaining_balance>=1.79923 & Custome...
2,c,0.105,38,Amount_vs_remaining_balance>=1.79923 & Custome...
3,d,0.062,162,Transaction_is_withdrawal==1 & Amount_vs_daily...
4,e,0.167,30,Amount_vs_remaining_balance>=1.79923 & large_d...
5,f,0.200,30,Amount_vs_remaining_balance>=1.79923 & Account...
6,g,0.074,27,Account_balance<=2.78403e+06 & Amount_vs_daily...
7,h,0.172,64,Transaction_is_withdrawal==1 & Account_one_mon...
8,i,0.096,239,Transaction_Amount_abs>=5.092e+07 & Transactio...
9,j,0.069,609,Amount_vs_remaining_balance>=1.79923 & Transac...


## 3-2. 결합 방식 2종 × precision 기준 4단계

| 방식 | 동작 |
|---|---|
| `override` | 규칙에 걸리면 ML 예측을 **덮어쓴다** |
| `boost` | 해당 클래스 확률에 `0.30 × precision` 을 **가산**한 뒤 argmax |


In [ ]:
def adopt(prec_min):
    out, seen = [], set()
    for a, cl, p, s_ in sorted(cands, key=lambda t: (-t[2], -t[3])):
        if p < prec_min or s_ < RULE_MIN_SUPPORT: continue
        d = _desc(a)
        if d in seen: continue
        seen.add(d); out.append({"atoms": a, "class": cl, "p": p, "desc": d})
    return out

ml_macro = macro
yb_ = yb; scan_rows = []
for pm in PRECISION_SCAN:
    rules = adopt(pm)
    if rules:
        po = pred.copy()
        for r in sorted(rules, key=lambda r: r["p"]):
            po[_cond_mask(X_va, r["atoms"])] = r["class"]
        Pb = proba.copy()
        ci = {c: i for i, c in enumerate(_cls)}
        for r in rules:
            m = _cond_mask(X_va, r["atoms"])
            if m.any(): Pb[np.ix_(m, [ci[r["class"]]])] += BOOST_WEIGHT * r["p"]
        pb = np.asarray(_cls)[Pb.argmax(axis=1)]
        o_m = f1_score(y_va, po, average="macro"); b_m = f1_score(y_va, pb, average="macro")
    else:
        o_m = b_m = ml_macro
    scan_rows.append({"precision 기준": pm, "채택 규칙수": len(rules),
                      "override ΔF1": round(o_m - ml_macro, 4),
                      "boost ΔF1": round(b_m - ml_macro, 4)})
_table(pd.DataFrame(scan_rows), "precision 기준별 규칙 수 & 성능 변화 (ML 대비)")

o_best = max(r["override ΔF1"] for r in scan_rows)
b_best = max(r["boost ΔF1"] for r in scan_rows)
o_worst = min(r["override ΔF1"] for r in scan_rows)
print(f"\n[결론]  순수 ML Macro-F1 = {ml_macro:.4f}")
if o_best <= 1e-6 and b_best <= 1e-6:
    print("  규칙 결합 순이득 없음 → ML 중심이 타당.")
    print(f"    · override: {o_worst:+.4f} (소폭 악화 또는 무변화)")
    print(f"    · boost   : {b_best:+.4f} (무변화)")
    print("  이유: 고정밀 규칙이 소수 유형에 한정되고, 그 유형은 ML이 이미 잘 분류함.")
else:
    print(f"  일부 방식에서 개선(override {o_best:+.4f} / boost {b_best:+.4f}) → 위 표 참고.")
print(f"  ※ 사기 비율 {yb_.mean():.2%} (극단 불균형) → 단순 규칙은 정밀도 한계")


[precision 기준별 규칙 수 & 성능 변화 (ML 대비)]


,precision 기준,채택 규칙수,override ΔF1,boost ΔF1
0,0.85,1,-0.004,0.000
1,0.70,3,-0.004,-0.004
2,0.50,3,-0.004,-0.004
3,0.30,3,-0.004,-0.004



[결론]  순수 ML Macro-F1 = 0.6138
  규칙 결합 순이득 없음 → ML 중심이 타당.
    · override: -0.0040 (소폭 악화 또는 무변화)
    · boost   : +0.0000 (무변화)
  이유: 고정밀 규칙이 소수 유형에 한정되고, 그 유형은 ML이 이미 잘 분류함.
  ※ 사기 비율 1.08% (극단 불균형) → 단순 규칙은 정밀도 한계


### 결론 — 규칙 결합은 순이득이 없다

| precision 기준 | 채택 규칙 | override ΔF1 | boost ΔF1 |
|---|---|---|---|
| 0.85 | 1개 | **−0.0040** | 0.0000 |
| 0.70 | 3개 | −0.0040 | −0.0040 |
| 0.50 | 3개 | −0.0040 | −0.0040 |
| 0.30 | 3개 | −0.0040 | −0.0040 |

**왜 안 되는가** — 유형별 최고 달성 가능 precision 을 보면 이유가 명확하다.
`a` 유형만 0.933 이고, 나머지는 전부 **0.2 미만**(`d` 0.062 · `j` 0.069 · `g` 0.074).
사기 비율이 1.08% 인 극단 불균형에서 단순 조건 조합으로는 정밀도가 나오지 않는다.

게다가 유일하게 쓸 만한 `a` 유형 규칙은 **ML 이 이미 Recall 0.957 로 잘 맞히는** 유형이다.
규칙이 보탤 것이 없다.

→ **ML 중심 유지.** 튜터 권고는 검토했고, 검토 결과 채택하지 않았다.
